In [ ]:
#import libraries

import scicone
import numpy as np
import pickle
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import  io
import scanpy as sc
import anndata
import pyranges
import gseapy as gp

# Set up SCICoNE
install_path = '/cluster/work/bewi/members/andress/pylabs/SCICoNE/build/'
install_path_local = "/home/andress/pylabs/SCICoNE_lab/build/"
temporary_outpath = './'

seed = 42 # for reproducibility

np.random.seed(seed)

# Create SCICoNE object
sci = scicone.SCICoNE(install_path_local, temporary_outpath, verbose=False)

In [ ]:


# Define paths
scdna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/cnv/'
scrna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19/'

scdna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/SA501/cnv'
scrna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19'

# Try SSH paths first
try:
    if os.path.exists(scdna_path_ssh) and os.path.exists(scrna_path_ssh):
        scdna_path = scdna_path_ssh
        scrna_path = scrna_path_ssh
    else:
        raise FileNotFoundError("SSH paths are not accessible.")
except FileNotFoundError:
    # Use local paths
    if os.path.exists(scdna_path_local) and os.path.exists(scrna_path_local):
        scdna_path = scdna_path_local
        scrna_path = scrna_path_local
    else:
        raise FileNotFoundError("Neither SSH nor local paths are accessible.")

In [ ]:
annot = sc.queries.biomart_annotations(
        "hsapiens",
        ["ensembl_gene_id", "start_position", "end_position", "chromosome_name"],
    ).set_index("ensembl_gene_id")

annot = annot.reset_index()
annot = annot.rename(columns={'start_position':'Start', 'end_position': 'End', 'chromosome_name': 'Chromosome'})

gr_annotations = pyranges.from_dict(annot.to_dict())

gr_annotations.head()

Construct Clusters based on mean, median and 

In [ ]:
# Load the sorted adata_clustered
adata_clustered_sorted = anndata.read_h5ad(f'{temporary_outpath}/adatas/adata_leiden.h5ad')
adata_clustered_sorted.shape

In [ ]:
# Get cluster labels
cluster_labels = adata_clustered_sorted.obs['leiden']
unique_clusters = cluster_labels.unique()

# Create meta cells with different aggregation methods: mean, median, and sum
meta_cells_mean = []  # Mean (average)
meta_cells_median = []  # Median (middle value)
meta_cells_sum = []  # Sum (total)
meta_cell_names = []

for cluster in unique_clusters:
    cluster_indices = np.where(cluster_labels == cluster)[0]
    cluster_data = adata_clustered_sorted.X[cluster_indices]

    # Compute aggregations
    cluster_mean = cluster_data.mean(axis=0)
    cluster_median = np.median(cluster_data, axis=0)
    cluster_sum = cluster_data.sum(axis=0)

    meta_cells_mean.append(cluster_mean)
    meta_cells_median.append(cluster_median)
    meta_cells_sum.append(cluster_sum)

    num_cells = len(cluster_indices)
    meta_cell_names.append(f'Cluster {cluster} ({num_cells} cells)')

# Convert to AnnData objects for each aggregation method
# 1. Mean meta-cells
meta_cells_matrix_mean = np.vstack(meta_cells_mean)
meta_adata_mean = anndata.AnnData(X=meta_cells_matrix_mean)
meta_adata_mean.obs['cluster'] = meta_cell_names
meta_adata_mean.obs_names = meta_cell_names
meta_adata_mean.var = adata_clustered_sorted.var.copy()

# 2. Median meta-cells
meta_cells_matrix_median = np.vstack(meta_cells_median)
meta_adata_median = anndata.AnnData(X=meta_cells_matrix_median)
meta_adata_median.obs['cluster'] = meta_cell_names
meta_adata_median.obs_names = meta_cell_names
meta_adata_median.var = adata_clustered_sorted.var.copy()

# 3. Sum meta-cells
meta_cells_matrix_sum = np.vstack(meta_cells_sum)
meta_adata_sum = anndata.AnnData(X=meta_cells_matrix_sum)
meta_adata_sum.obs['cluster'] = meta_cell_names
meta_adata_sum.obs_names = meta_cell_names
meta_adata_sum.var = adata_clustered_sorted.var.copy()

# Save the meta cells
meta_adata_mean.write_h5ad(f'{temporary_outpath}/adatas/adata_clusters_mean.h5ad')
meta_adata_median.write_h5ad(f'{temporary_outpath}/adatas/adata_clusters_median.h5ad')
meta_adata_sum.write_h5ad(f'{temporary_outpath}/adatas/adata_clusters_sum.h5ad')

print("Meta cells created and saved from sorted adata_clustered.")

In [ ]:
# # Check if cluster_file.X is formatted like filtered_counts
# for cluster_file in [mean_clusters, median_clusters, sum_clusters]:
#     print(f"Checking cluster_file: {cluster_file}")
#     print(f"Type of cluster_file.X: {type(cluster_file.X)}")
#     print(f"Shape of cluster_file.X: {cluster_file.X.shape}")
#     print(f"Content of cluster_file.X (first 5 rows):\n{cluster_file.X[:5]}")
#     print("\n")

In [ ]:
# Sort adata.var by 'Chromosome' and 'End'

sum_clusters = meta_adata_sum
sum_clusters.var['Chromosome'] = sum_clusters.var['Chromosome'].astype(str)
sorted_var = sum_clusters.var.sort_values(by=['Chromosome', 'End'])

# Reorder adata.X columns based on the sorted var index
sum_clusters = sum_clusters[:, sorted_var.index]

# Verify the sorting
print(sum_clusters.var[['Chromosome', 'End']].head())